# **Text-to-SQL Project Proof of Concept**

This Colab notebook contains an experimental implementation of the proposed approach of our Text-to-SQL pipeline. The notebook contains the following sections:

1. Neccessary Set-up
2. Data Storing
3. Tool Creation
4. Agent Building
5. Task Definition
6. Multi-agent Workflow Creation
7. App Testing & Output Verification




**Neccessary Set-up**

Installation of Frameworks & Libraries

In [1]:
%pip install crewai -qq
%pip install langchain_chroma -qq
%pip install langchain_openai -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.0/235.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.0/134.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.5 MB/s eta 0:

Setting-up the Base LLM for Agents

In [25]:
import os
from google.colab import userdata
from crewai import LLM


# Retrieve the OpenAI API key from Google Colab's user data storage
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Initialize the base LLM model (GPT-4) for agents to use in CrewAI
llm = LLM(model="gpt-4")

In [3]:
from google.colab import files

# Upload Data files
for i in range(2):
  uploaded = files.upload()


Saving Dataset-Mental-Disorders.csv to Dataset-Mental-Disorders.csv


Saving cleaned_houses_info.csv to cleaned_houses_info.csv


**Data Storing**

This code section creates an SQLite database (multi_domain.db), loads two example CSV files (Dataset-Mental-Disorders.csv and cleaned_houses_info.csv), creates two separate tables (mental_disorders and real_estate) in the database, and stores the corresponding data into these tables.

In [4]:
import pandas as pd
from sqlalchemy import create_engine, Column, Integer, Float, String, MetaData, Table
from crewai.tools import BaseTool




# Load CSV files
df_psychology = pd.read_csv("Dataset-Mental-Disorders.csv")
df_real_estate = pd.read_csv("cleaned_houses_info.csv")


# Rename columns in psychology DataFrame to remove spaces
df_psychology.rename(columns=lambda x: x.strip().replace(" ", "_"), inplace=True)
# For example, "Patient Number" becomes "Patient_Number"


# Add unique primary key identifiers
df_psychology['patient_id'] = range(1, df_psychology.shape[0] + 1)
df_real_estate['property_id'] = range(1, df_real_estate.shape[0] + 1)


# Create SQLite engine
engine = create_engine('sqlite:///multi_domain.db')

# Define the database schema
metadata = MetaData()

# Psychology Table (using new column names)
mental_disorders = Table(
    'mental_disorders', metadata,
    Column('patient_id', Integer, primary_key=True),
    Column('Patient_Number', String),
    Column('Sadness', String),
    Column('Euphoric', String),
    Column('Exhausted', String),
    Column('Sleep_dissorder', String),
    Column('Mood_Swing', String),
    Column('Suicidal_thoughts', String),
    Column('Anorxia', String),
    Column('Authority_Respect', String),
    Column('Try-Explanation', String),  # If you prefer to change, you can rename this as well
    Column('Aggressive_Response', String),
    Column('Ignore_&_Move-On', String),
    Column('Nervous_Break-down', String),
    Column('Admit_Mistakes', String),
    Column('Overthinking', String),
    Column('Sexual_Activity', String),
    Column('Concentration', String),
    Column('Optimisim', String),
    Column('Expert_Diagnose', String)
)

# Real Estate Table
real_estate = Table(
    'real_estate', metadata,
    Column('property_id', Integer, primary_key=True),
    Column('bedrooms', Integer),
    Column('bathrooms', Integer),
    Column('living_space', Float),
    Column('address', String),
    Column('city', String),
    Column('state', String),
    Column('zipcode', Integer),
    Column('latitude', Float),
    Column('longitude', Float),
    Column('property_url', String),
    Column('price', Float)
)

# Create tables in the database
metadata.create_all(engine)

# Insert data into the tables
df_psychology.to_sql('mental_disorders', engine, if_exists='replace', index=False)
df_real_estate.to_sql('real_estate', engine, if_exists='replace', index=False)




535

**Tool Creation**

This VectorSearchTool performs semantic search between the user query and database schema information using vector embeddings. It enables intelligent retrieval of relevant table and column names based on a given query.

In [29]:
class VectorSearchTool(BaseTool):
    name: str = "VectorSearchTool"
    description: str = "Search for relevant information using vector search"


    def _run(self, query: str) -> str:

      from langchain_chroma import Chroma
      from langchain_openai import OpenAIEmbeddings
      import json


      # Extract schema information
      # We'll use M-schema structure for the final pipeline
      schema_info = []
      for table in metadata.tables.values():
          table_name = table.name
          for column in table.columns:
              schema_info.append(f"{table_name}.{column.name}")

      #print("Schema Information:", schema_info)


      # Initialize OpenAI embeddings
      embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

      # Create Chroma vector store
      vectorstore = Chroma.from_texts(
          texts=schema_info,  # Schema information as text
          embedding=embeddings,  # OpenAI embeddings
          collection_name="schema_knowledge_base"  # Name of the collection
      )

      # Create a retriever for semantic search
      retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 10})

      #print("Schema Knowledge Base Created in Chroma.")


      # Perform semantic search
      results = retriever.get_relevant_documents(query)

      # Display results
      relevant_schema_info = [result.page_content for result in results]


      # Extract table names
      table_names = set()  # Use a set to avoid duplicates
      for result in results:
          table_name = result.page_content.split(".")[0]  # Split at "." and take the first part
          table_names.add(table_name)

      # Format output as JSON string
      output = {
        "query": query,
        "matching_schema": relevant_schema_info,
        "inferred_tables": list(table_names)
    }

      return json.dumps(output, indent=2)


vector_search_tool = VectorSearchTool()

#testing the tool
#vector_search_tool._run(query="what symptoms does a patient with depression has")


# Example query
      #query = "Find columns related to patient mental health."
      #query = "what symptoms does a patient with depression has"
      #query = "In which state in the US i can buy a house under 100000"


This SQLiteTool is designed to be utilized by the data_retrieving_agent to execute the Generated SQL query, retrieve data and return the data in a pandas dataframe format

In [6]:
class SQLiteTool(BaseTool):
    name: str = "SQLiteTool"
    description: str = "Executes SQL queries on an SQLite database"

    def _run(self, query: str) -> str:
        import pandas as pd

        # Execute the query using the engine
        result_df = pd.read_sql(query, engine)

        # Convert the DataFrame result to a string (or any format you need)
        return result_df

# Instantiate the tool with the engine that was created and used above
sql_search_tool = SQLiteTool()




###############  Testing the Tool   #########################

"""
# Query example: Fetch all real estate listings
query = "SELECT * FROM real_estate"
result = sql_search_tool._run(query)

result
"""

'\n# Query example: Fetch all real estate listings\nquery = "SELECT * FROM real_estate"\nresult = sql_search_tool._run(query)\n\nresult\n'

**Agent Building**

In [7]:
# Import crewai classes for agent, task and crew creation
from crewai import Agent, Task, Crew

This section contains all of the Agent's definitions and descriptions.

1. vector_search_agent : This agent takes the user's natural language query, performs semantic search on the schema knowledge base, and retrieves relevant table schemas and table names using the VectorSearchTool.

2. SQL Generation Agent : This agent converts the user's query and the retrieved schema information into a syntactically correct and highly accurate SQL query. It ensures that the query is well-optimized and effectively retrieves the required data. The generated SQL query is then passed to the Data Retrieving Agent.

3. Data Retrieving Agent : This agent validates the SQL query's syntax, executes it using the SQLiteTool, and retrieves the requested data from the database. It ensures that the SQL query is error-free before running it. The retrieved data is then forwarded to the Answer Generation Agent.

4. Answer Generation Agent : This agent takes the user's original query and the retrieved data to generate a concise and clear natural language response. It ensures that the response is both accurate and well-supported by the retrieved data. The final output includes both the natural language answer and the relevant supporting data.





In [40]:
# Agent - 1
vector_search_agent = Agent(
    role="Vector Search Agent",
    goal="Perform a semantic search between the user's '{query}' and the schema knowledge base to retrieve relevant table schemas and table names.",
    backstory="""You are a highly skilled Semantic Search Engineer with expertise in natural language processing and vector-based search techniques.
                Your task is to understand the query, perform a similarity search against the schema knowledge base using the vector_search_tool,
                and retrieve the most relevant table schemas and table names. You will then pass the json object containing query, retrived tables schemas, and
                table names to the sql_generation_agent.""",
    llm=llm,
    tools=[vector_search_tool],
    allow_delegation=False,
    verbose=True
)



# Agent - 2
sql_generation_agent = Agent(
    role="SQL Generation Agent",
    goal="Generate syntactically correct and highly accurate SQL query that answer the user's '{query}' using the provided relevant schema information and table names.",
    backstory="""You are an experienced SQL query generation expert with deep knowledge of relational databases and SQL syntax.
                Your task is to translate the user's natural language query into a SQL query that accurately retrieve the requested data.
                You will receive the user's '{query}' along with the relevant table schema information and table names returned by the vector search agent.
                The output from the vector search agent is provided as a json object in the format # Format output as JSON string.

                Note: Think step by step, visualize each state after taking each step.
    output = {
        "query": query,
        "matching_schema": relevant_schema_info,
        "inferred_tables": list(table_names)
    }

    .
                Use this information to construct SQL query that are both syntactically correct and optimized for performance.
                Pass the genarted sql query and the user_query to the data_retrieving_agent.""",
    llm=llm,
    allow_delegation=False,
    verbose=True
)




# Agent - 3
data_retrieving_agent = Agent(
    role="Data Retrieving Agent",
    goal="Validate the SQL query's syntax generated by the SQL Generation Agent, execute it using the sql_search_tool, and return the retrieved data.",
    backstory="""You are a seasoned data retrieval expert with a deep understanding of SQL syntax and database querying.
                Your responsibility is to ensure that the SQL query provided is syntactically correct before executing it.
                After validation, you will use the sql_search_tool to execute the query on the target database and then return the retrieved data in a clear, formatted manner.
                Pass the retrived data and the user query to the answer_generating_agent.""",
    llm=llm,
    tools=[sql_search_tool],  # sql_search_tool should be the tool used for executing SQL queries.
    allow_delegation=False,
    verbose=True
)




# Agent - 4
answer_generation_agent = Agent(
    role="Answer Generation Agent",
    goal="Generate a clear, natural language answer to the user's query using both the original query and the retrieved data, and return the answer along with the supporting data.",
    backstory="""You are an expert in both data analysis and natural language generation. Your task is to take the user's query and the data retrieved from the database,
                and synthesize a concise, accurate, and human-readable answer. In addition to the natural language response, you also provide the supporting data that justifies your answer.
                This ensures that your response is not only clear but also backed by concrete data.
                Return the natural language response and the answer-relevant data.""",
    llm=llm,
    allow_delegation=False,
    verbose=True
)



**Task Definition**

This section contains all of the Agent's task definitions.

1. Vector Search Task: Performs a semantic search on the schema knowledge base to retrieve relevant table schemas and names based on the user's query. Outputs a list of matching schema details and passes them to the SQL Generation Agent.

2. SQL Generation Task: Converts the user query and schema details into a syntactically correct and optimized SQL query to fetch the required data. Passes the SQL query to the Data Retrieving Agent.

3. Data Retrieving Task: Validates and executes the SQL query on the database using SQLiteTool, then returns the retrieved structured data to the Answer Generation Agent.

4. Answer Generation Task: Analyzes the retrieved data and synthesizes a natural language response, ensuring clarity and supporting the answer with relevant data.

In [41]:
# Task for Agent - 1: Vector Search Task
vector_search_task = Task(
    description=(
        "1. Receive the user's natural language query.\n"
        "2. Perform a semantic similarity search against the schema knowledge base using the vector_search_tool.\n"
        "3. Retrieve the most relevant table schemas and table names in the format [relevant_schema_info, list(table_names)].\n"
        "4. Pass the user's query along with the retrieved schema information and table names to the SQL Generation Agent."
    ),
    expected_output=(
        "A list containing the relevant schema information and table names, e.g., [relevant_schema_info, list(table_names)]."
    ),
    agent=vector_search_agent,
    human_input=True
)

# Task for Agent - 2: SQL Generation Task
sql_generation_task = Task(
    description=(
        "1. Receive the user's query and the output from the Vector Search Agent (in the format [relevant_schema_info, list(table_names)]).\n"
        "2. Translate the natural language query into a syntactically correct SQL query that accurately retrieves the requested data.\n"
        "3. Optimize the SQL query for performance if possible.\n"
        "4. Pass the generated SQL query along with the user query to the Data Retrieving Agent."
    ),
    expected_output=(
        "A syntactically correct SQL query that can retrieve the requested data from the database."
    ),
    agent=sql_generation_agent,
    human_input=False
)

# Task for Agent - 3: Data Retrieving Task
data_retrieving_task = Task(
    description=(
        "1. Receive the SQL query and the user query from the SQL Generation Agent.\n"
        "2. Validate the syntax of the SQL query.\n"
        "3. Execute the SQL query on the target database using the sql_search_tool.\n"
        "4. Return the retrieved data in a clear, formatted manner along with the user query to the Answer Generation Agent."
    ),
    expected_output=(
        "A formatted dataset (e.g., a pandas DataFrame) containing the results of the executed SQL query."
    ),
    agent=data_retrieving_agent,
    human_input=False
)

# Task for Agent - 4: Answer Generation Task
answer_generation_task = Task(
    description=(
        "1. Receive the user's original query and the data retrieved by the Data Retrieving Agent.\n"
        "2. Analyze the data alongside the user query to synthesize a clear, natural language answer that addresses the request.\n"
        "3. Provide supporting data that justifies and backs the generated answer.\n"
        "4. Return the natural language answer along with the answer-relevant supporting data."
    ),
    expected_output=(
        "A natural language answer to the user's query, accompanied by the supporting data used to generate the answer."
    ),
    agent=answer_generation_agent,
    human_input=False
)


**Multi-agent Workflow Creation**

This section creates puts all of the pipeline components together to create the Multi-agentic workflow

In [42]:
text_to_sql_crew = Crew(
    agents=[vector_search_agent, sql_generation_agent, data_retrieving_agent, answer_generation_agent],
    tasks=[vector_search_task, sql_generation_task, data_retrieving_task, answer_generation_task],
    verbose=True,
    process_type="sequential")

Kickoff the multi-agentic crew pipeline

In [43]:
query = input("Enter your query here:")

vector_search_task.description = f"Here is the user's natural language query: {query} now, "

result = text_to_sql_crew.kickoff()

#what symptoms and diagnosis the patient number 3 has?
#Are there any properties priced below $800,000 with at least 3 bedrooms?

Enter your query here:what symptoms and diagnosis the patient number 3 has?
# Agent: Vector Search Agent
## Task: Here is the user's natural language query: what symptoms and diagnosis the patient number 3 has? now, 


# Agent: Vector Search Agent
## Thought: Based on the given query, the user wants to know the symptoms and diagnosis related to a specific patient (patient number 3). According to this, the relevant information might be in formations related to patient data, like unique identifiers (patient numbers), symptoms and diagnosis. I will perform a vector search against the schema knowledge base to find this information.
## Using tool: VectorSearchTool
## Tool Input: 
"{\"query\": \"what symptoms and diagnosis the patient number 3 has\"}"
## Tool Output: 
{
  "query": "what symptoms and diagnosis the patient number 3 has",
  "matching_schema": [
    "mental_disorders.Patient_Number",
    "mental_disorders.Patient_Number",
    "mental_disorders.Patient_Number",
    "mental_disord

**Verification of the App output**

##############      Response using M-schema Format     ###################

Mental Disorders Table

**Input-1:** what symptoms and diagnosis the patient number 3 has?
**Final Output:** ## Final Answer:
According to the database, there are currently no patients who are experiencing both sadness and suicidal thoughts simultaneously.

Supporting Data: The query "SELECT * FROM mental_disorders WHERE Sadness = 'Yes' AND Suicidal_thoughts = 'Yes';" did not return any results. The retrieved data was empty.


verification of Output-1

In [22]:
psy_df = pd.read_csv("Dataset-Mental-Disorders.csv")
psy_df[psy_df['Patient Number'] == "Patiant-03"]


###### The system responded with a wrong answer ######

,Patient Number,Sadness,Euphoric,Exhausted,Sleep dissorder,Mood Swing,Suicidal thoughts,Anorxia,Authority Respect,Try-Explanation,Aggressive Response,Ignore & Move-On,Nervous Break-down,Admit Mistakes,Overthinking,Sexual Activity,Concentration,Optimisim,Expert Diagnose
2,Patiant-03,Sometimes,Most-Often,Sometimes,Sometimes,YES,NO,NO,NO,YES,YES,NO,YES,YES,NO,6 From 10,5 From 10,7 From 10,Bipolar Type-1


 Real Estate Table

**Input**: Are there any properties priced below $800,000 with at least 3 bedrooms?
**Output**: BadRequestError

#################   Response using Our <table_name.column_name> schema format  #################


Real Estate Table



**Input**: Are there any properties priced below $800,000 with at least 3 bedrooms?
**Output**: ## Final Answer:
The database has returned a total of 287 properties that match your criteria of having 3 or more bedrooms and priced less than 800,000. For instance, a property at '2538 Elwood Ave, South Lake Tahoe, CA 96150' is priced at 699,000, has 4 bedrooms and 2 bathrooms, with a living space of 1690 square feet. Another property at '4334 Valley St, Cottonwood, CA 96022', priced at 484,900, offers 3 bedrooms and 2 bathrooms within a 2130 square feet living space. You may explore other properties that fit your requirements in the provided dataset.

Supporting Data:
1. '2538 Elwood Ave, South Lake Tahoe, CA 96150'
Price: 699,000
Bedrooms: 4
Bathrooms: 2
Living Space: 1690 square feet
[Property Link](https://www.zillow.com/homedetails/2538-Elwood-Ave-South-Lake-Tahoe-CA-96150/17292821_zpid/)

2. '4334 Valley St, Cottonwood, CA 96022'
Price: 484,900
Bedrooms: 3
Bathrooms: 2
Living Space: 2130 square feet
[Property Link](https://www.zillow.com/homedetails/4334-Valley-St-Cottonwood-CA-96022/15245388_zpid/)

Additional 285 properties available in the database.


In [34]:
real_df = pd.read_csv("cleaned_houses_info.csv")
real_df.head()

######### the system couldn't give any answer #########

,bedrooms,bathrooms,living_space,address,city,state,zipcode,latitude,longitude,property_url,price
0,3,3,1706.0,"10615 Sara Bear Ln, Truckee, CA 96161",Truckee,CA,96161,39.322980,-120.174446,https://www.zillow.com/homedetails/10615-Sara-...,1275000.0
1,5,3,2142.0,"11553 E Ridge Rd, Truckee, CA 96161",Truckee,CA,96161,39.335030,-120.157870,https://www.zillow.com/homedetails/11553-E-Rid...,949900.0
2,4,2,2362.0,"11265 Mount Rose View Dr, Truckee, CA 96161",Truckee,CA,96161,39.363483,-120.151610,https://www.zillow.com/homedetails/11265-Mount...,1475000.0
3,4,2,1552.0,"11862 Rio Vista Dr, Truckee, CA 96161",Truckee,CA,96161,39.315920,-120.194115,https://www.zillow.com/homedetails/11862-Rio-V...,850000.0
4,4,2,1690.0,"2538 Elwood Ave, South Lake Tahoe, CA 96150",South Lake Tahoe,CA,96150,38.922966,-119.984344,https://www.zillow.com/homedetails/2538-Elwood...,699000.0


In [35]:
filtered_df = real_df[(real_df['price'] < 800000.0) & (real_df['bedrooms'] >= 3)]
filtered_df


######### The system responded with the correct answer ########

,bedrooms,bathrooms,living_space,address,city,state,zipcode,latitude,longitude,property_url,price
4,4,2,1690.0,"2538 Elwood Ave, South Lake Tahoe, CA 96150",South Lake Tahoe,CA,96150,38.922966,-119.984344,https://www.zillow.com/homedetails/2538-Elwood...,699000.0
6,3,2,1104.0,"2530 William Ave, South Lake Tahoe, CA 96150",South Lake Tahoe,CA,96150,38.923290,-119.985160,https://www.zillow.com/homedetails/2530-Willia...,642000.0
7,3,2,1710.0,"2148 Albert Ave, South Lake Tahoe, CA 96150",South Lake Tahoe,CA,96150,38.925377,-120.015240,https://www.zillow.com/homedetails/2148-Albert...,799000.0
12,3,2,1196.0,"803 Michael Dr, South Lake Tahoe, CA 96150",South Lake Tahoe,CA,96150,38.925760,-119.999420,https://www.zillow.com/homedetails/803-Michael...,739000.0
16,3,2,1150.0,"2525 Armstrong Ave, South Lake Tahoe, CA 96150",South Lake Tahoe,CA,96150,38.922490,-119.984520,https://www.zillow.com/homedetails/2525-Armstr...,399000.0
...,...,...,...,...,...,...,...,...,...,...,...
529,3,2,1224.0,"22150 Oak Run Pl, Cottonwood, CA 96022",Cottonwood,CA,96022,40.357570,-122.211850,https://www.zillow.com/homedetails/22150-Oak-R...,324900.0
530,3,3,2000.0,"11300 N Bloomfield Graniteville Rd, Nevada Cit...",Nevada City,CA,95959,39.281124,-121.014050,https://www.zillow.com/homedetails/11300-N-Blo...,525000.0
532,3,3,2448.0,"16865 Big Pines Rd, Cottonwood, CA 96022",Cottonwood,CA,96022,40.285110,-122.404274,https://www.zillow.com/homedetails/16865-Big-P...,637500.0
533,3,2,1560.0,"16310 Bowman Rd, Cottonwood, CA 96022",Cottonwood,CA,96022,40.324970,-122.432640,https://www.zillow.com/homedetails/16310-Bowma...,325000.0


Mean house price in Anderson: $527993.58


Mean house price in Dorris: $3357958.25
